In [1]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import zero_one_loss, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import ConfusionMatrixDisplay

In [2]:
# Task 1: Data Preparation and Reshaping

# Task 1a: Load the table and familiarize with contents
print("=== Task 1a: Loading and Exploring Data ===")

# Load the grambank dataset
df = pd.read_csv('grambank.csv')

# Display basic information about the dataset
print(f"Dataset shape: {df.shape}")
print(f"Number of languages: {df.shape[0]}")
print(f"Number of features + macroarea column: {df.shape[1]}")

# Display first few rows
print("\nFirst 5 rows:")
print(df.head())

# Check column names
print(f"\nColumn names (first 10): {df.columns[:10].tolist()}")
print(f"Last column (macroarea): {df.columns[-1]}")

# Check for null values
print(f"\nNull values per column (first 10 features):")
print(df.iloc[:, :10].isnull().sum())

# Check unique macroareas
print(f"\nUnique macroareas: {df.iloc[:, -1].unique()}")
print(f"Number of macroareas: {df.iloc[:, -1].nunique()}")

# Check data types
print(f"\nData types (first 10 features):")
print(df.dtypes[:10])

print("\n" + "="*50 + "\n")




=== Task 1a: Loading and Exploring Data ===
Dataset shape: (2463, 197)
Number of languages: 2463
Number of features + macroarea column: 197

First 5 rows:
       Language    GB020   GB021    GB022    GB023  GB024  GB025   GB026  \
0      'Are'are  present  absent   absent  present  Num-N  N-Dem  absent   
1          A'ou  present  absent  present   absent  Num-N  N-Dem  absent   
2         Abadi      NaN     NaN      NaN      NaN  N-Num  Dem-N  absent   
3          Abau   absent  absent   absent   absent  N-Num  N-Dem  absent   
4  Abenlen Ayta   absent  absent   absent   absent  Num-N  Dem-N     NaN   

     GB027    GB028  ...   GB422   GB430   GB431   GB432    GB433    GB519  \
0  present  present  ...  absent  absent  absent  absent  present  present   
1   absent   absent  ...     NaN  absent  absent  absent   absent  present   
2   absent  present  ...     NaN  absent  absent  absent  present      NaN   
3  present   absent  ...     NaN     NaN     NaN     NaN      NaN      NaN  

In [3]:
# Task 1b: Generate binarised version using get_dummies
print("=== Task 1b: Binarizing the Dataset ===")

# Separate features from macroarea column
features_df = df.iloc[:, :-1]  # All columns except the last one
macroarea_column = df.iloc[:, -1]  # Last column (macroarea)

# Apply get_dummies to the features (using default parameters)
# This will create separate columns for presence/absence of each feature
# and missing values will be indicated by zeros in both columns
features_binarized = pd.get_dummies(features_df)

print(f"Original features shape: {features_df.shape}")
print(f"Binarized features shape: {features_binarized.shape}")
print(f"Sample of binarized column names: {features_binarized.columns[:10].tolist()}")

print("\n" + "="*50 + "\n")


=== Task 1b: Binarizing the Dataset ===
Original features shape: (2463, 196)
Binarized features shape: (2463, 2861)
Sample of binarized column names: ["Language_'Are'are", "Language_A'ou", 'Language_Abadi', 'Language_Abau', 'Language_Abenlen Ayta', 'Language_Abipon', 'Language_Abkhaz', "Language_Abu' Arapesh", 'Language_Abui', 'Language_Abun']




In [4]:
# Task 1c: Prepare feature matrix X and label vector y
print("=== Task 1c: Preparing Feature Matrix and Labels ===")

# Convert binarized features to numpy array and cast to integers
X = features_binarized.values.astype(int)
print(f"Feature matrix X shape: {X.shape}")
print(f"Feature matrix X dtype: {X.dtype}")

# Convert macroarea labels to array of strings, then encode to integers
y_strings = macroarea_column.values.astype(str)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y_strings)

print(f"Label vector y shape: {y.shape}")
print(f"Label vector y dtype: {y.dtype}")
print(f"Label mapping: {dict(zip(label_encoder.classes_, range(len(label_encoder.classes_))))}")

print("\n" + "="*50 + "\n")



=== Task 1c: Preparing Feature Matrix and Labels ===
Feature matrix X shape: (2463, 2861)
Feature matrix X dtype: int64
Label vector y shape: (2463,)
Label vector y dtype: int64
Label mapping: {'Africa': 0, 'Australia': 1, 'Eurasia': 2, 'North America': 3, 'Papunesia': 4, 'South America': 5}




In [5]:
# Task 1d: Prepare train/test splits
print("=== Task 1d: Creating Train/Test Splits ===")

# Split into 80% training and 20% test data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set shape: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Test set shape: X_test={X_test.shape}, y_test={y_test.shape}")

# Check class distribution in train and test sets
print(f"\nClass distribution in training set:")
unique_train, counts_train = np.unique(y_train, return_counts=True)
for class_id, count in zip(unique_train, counts_train):
    class_name = label_encoder.inverse_transform([class_id])[0]
    print(f"  {class_name}: {count} ({count/len(y_train)*100:.1f}%)")

print(f"\nClass distribution in test set:")
unique_test, counts_test = np.unique(y_test, return_counts=True)
for class_id, count in zip(unique_test, counts_test):
    class_name = label_encoder.inverse_transform([class_id])[0]
    print(f"  {class_name}: {count} ({count/len(y_test)*100:.1f}%)")

print("\nTask 1 completed successfully!")
print("Ready to proceed to Task 2 (Decision Tree)")

=== Task 1d: Creating Train/Test Splits ===
Training set shape: X_train=(1970, 2861), y_train=(1970,)
Test set shape: X_test=(493, 2861), y_test=(493,)

Class distribution in training set:
  Africa: 450 (22.8%)
  Australia: 115 (5.8%)
  Eurasia: 450 (22.8%)
  North America: 194 (9.8%)
  Papunesia: 582 (29.5%)
  South America: 179 (9.1%)

Class distribution in test set:
  Africa: 112 (22.7%)
  Australia: 28 (5.7%)
  Eurasia: 113 (22.9%)
  North America: 49 (9.9%)
  Papunesia: 146 (29.6%)
  South America: 45 (9.1%)

Task 1 completed successfully!
Ready to proceed to Task 2 (Decision Tree)


In [6]:
# Task 2: Exploring the Data with a Decision Tree

# Task 2a: Fit a decision tree classifier and make predictions
print("=== Task 2a: Fitting Decision Tree Classifier ===")

# Create and fit decision tree with default settings
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

# Make predictions on test data
y_pred_dt = dt_classifier.predict(X_test)

print("Decision tree fitted successfully!")
print(f"Number of test samples: {len(y_test)}")
print(f"Number of predictions: {len(y_pred_dt)}")

print("\n" + "="*50 + "\n")

=== Task 2a: Fitting Decision Tree Classifier ===
Decision tree fitted successfully!
Number of test samples: 493
Number of predictions: 493




In [7]:
# Task 2b: Compute accuracy using zero_one_loss
print("=== Task 2b: Computing Accuracy ===")

# Use zero_one_loss to compute accuracy
# Note: zero_one_loss gives the fraction of misclassifications
# So accuracy = 1 - zero_one_loss
zero_one_error = zero_one_loss(y_test, y_pred_dt)
accuracy_dt = 1 - zero_one_error

print(f"Zero-one loss (error rate): {zero_one_error:.4f}")
print(f"Accuracy: {accuracy_dt:.4f} ({accuracy_dt*100:.2f}%)")

# Alternative way to verify accuracy
accuracy_verify = accuracy_score(y_test, y_pred_dt)
print(f"Accuracy (verification): {accuracy_verify:.4f}")

# Show some example predictions vs actual
print(f"\nFirst 10 predictions vs actual:")
print("Predicted | Actual    | Correct?")
print("-" * 35)
for i in range(min(10, len(y_test))):
    pred_name = label_encoder.inverse_transform([y_pred_dt[i]])[0]
    actual_name = label_encoder.inverse_transform([y_test[i]])[0]
    correct = "✓" if y_pred_dt[i] == y_test[i] else "✗"
    print(f"{pred_name:9} | {actual_name:9} | {correct}")

print("\n" + "="*50 + "\n")


=== Task 2b: Computing Accuracy ===
Zero-one loss (error rate): 0.3732
Accuracy: 0.6268 (62.68%)
Accuracy (verification): 0.6268

First 10 predictions vs actual:
Predicted | Actual    | Correct?
-----------------------------------
Eurasia   | South America | ✗
Papunesia | Eurasia   | ✗
South America | Papunesia | ✗
North America | North America | ✓
Papunesia | Papunesia | ✓
Papunesia | Papunesia | ✓
Australia | Africa    | ✗
North America | Papunesia | ✗
Africa    | Africa    | ✓
Papunesia | Papunesia | ✓


